# 📱 Liquidity Stress Prediction Challenge - Starter Notebook

Welcome to the starter notebook for this challenge.

In this challenge, your goal is to build a machine learning model that predicts whether a customer is likely to experience liquidity stress, using historical mobile money behaviour.

This notebook walks through a simple end-to-end pipeline:

- Loading and exploring the data
- Preprocessing
- Training a baseline model
- Evaluating performance
- Generating predictions for submission

The baseline model used here is **Logistic Regression**, which is a good starting point for tabular classification problems.


In [ ]:
# 📚 Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


sns.set_style("whitegrid")
pd.set_option("display.max_columns", 200)


In [ ]:
# 📥 Load the Data
train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")

SUBMISSION_TARGET = "Target"

# Create a sample submission template directly from the test IDs
sample_submission = pd.DataFrame({
    "ID": test["ID"],
    SUBMISSION_TARGET: 0.5
})

# Peek at the data
train.head()


## 🔍 Exploratory Data Analysis (EDA)

Let’s explore the training data to understand the features and target.


In [ ]:
train.info()

In [ ]:
# Check missing values
missing_summary = train.isnull().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0].head(20)


### 🏷️ Encoding Categorical Variables

We use **one-hot encoding** to convert categorical columns into numeric format so they can be used by the model.


In [ ]:
# Identify the ID column, target column and feature columns
TARGET = "liquidity_stress_next_30d"
ID_COL = "ID"

feature_cols = [c for c in train.columns if c not in [ID_COL, TARGET]]
cat_cols = [c for c in feature_cols if train[c].dtype == "object"]

cat_cols


In [ ]:
# Apply one-hot encoding to the feature columns only
X_raw = train[feature_cols].copy()
test_raw = test[feature_cols].copy()

X_encoded = pd.get_dummies(X_raw, columns=cat_cols, drop_first=True)
test_encoded = pd.get_dummies(test_raw, columns=cat_cols, drop_first=True)

# Align columns so train and test have the same encoded feature columns
X_encoded, test_encoded = X_encoded.align(test_encoded, join="left", axis=1, fill_value=0)

X_encoded.head()


In [ ]:
# Fill missing numeric values with the median from the training data
for col in X_encoded.columns:
    median_value = X_encoded[col].median()
    X_encoded[col] = X_encoded[col].fillna(median_value)
    test_encoded[col] = test_encoded[col].fillna(median_value)

print("Remaining missing values in train:", X_encoded.isnull().sum().sum())
print("Remaining missing values in test :", test_encoded.isnull().sum().sum())


In [ ]:
# Distribution of target variable
sns.countplot(x=TARGET, data=train)
plt.title("Liquidity Stress Distribution")
plt.show()


In [ ]:
# Plot distributions of a few useful numeric features
num_features = [
    "arpu",
    "age",
    "x_90_d_activity_rate",
    "m1_daily_avg_bal",
    "m1_withdraw_total_value",
    "m1_received_total_value"
]

for col in num_features:
    if col in train.columns:
        sns.histplot(train[col], kde=True)
        plt.title(f"Distribution of {col}")
        plt.show()


## 🧹 Preprocessing

We’ll split the training data into features and target, then create a validation set.


In [ ]:
# Features and target
X = X_encoded
y = train[TARGET]

# Split into train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_val.shape


In [ ]:
print(f"Training with {X.shape[1]} encoded features.")


## 🤖 Model Training

We’ll use a Logistic Regression classifier as our baseline model.


In [ ]:
# Train a Logistic Regression model
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=100,
                                class_weight="balanced",
                                solver="liblinear")
    )
])

pipeline.fit(X_train, y_train)

In [ ]:
# Predict on validation set
val_pred_prob = pipeline.predict_proba(X_val)[:, 1]
val_pred_label = (val_pred_prob >= 0.5).astype(int)

# Evaluate
print("Validation ROC-AUC:", roc_auc_score(y_val, val_pred_prob))
print("\nClassification Report:")
print(classification_report(y_val, val_pred_label))


In [ ]:
# 📈 Evaluate Model on Validation Set
cm = confusion_matrix(y_val, val_pred_label)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()


## 📊 Optional: Feature Importance

This section gives a quick sense of which features are most useful to the baseline model. It is optional and can be skipped if you only want to generate a first submission quickly.


In [ ]:
from sklearn.inspection import permutation_importance


result = permutation_importance(
    pipeline,
    X_val,
    y_val,
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

importances = pd.Series(result.importances_mean, index=X.columns)

(
    importances
    .sort_values(ascending=False)
    .head(25)
    .sort_values()
    .plot(kind="barh", figsize=(10,8))
)

plt.title("Top 25 Feature Importance (Permutation Importance)")
plt.xlabel("Mean Decrease in Model Score")
plt.show()


## 🚀 Predictions on Test Set

Let’s predict on the test set and generate a submission file.

The submission file must contain two columns:

- `ID`: the customer ID from the test set
- `Target`: the predicted probability that the customer will experience liquidity stress in the next 30 days

`Target` should be a probability between 0 and 1, not a hard class label.


In [ ]:
# Retrain the model on the full training set
pipeline.fit(X, y)

In [ ]:
# Check that train and test feature columns match before predicting
assert list(test_encoded.columns) == list(X.columns)
print("Train and test feature columns match.")


In [ ]:
# Use the encoded test features for prediction
test_features = test_encoded.copy()
test_features.head()


In [ ]:
# Predict probabilities on test data
test_predictions =  pipeline.predict_proba(test_features)[:, 1]

In [ ]:
# Prepare submission
submission = sample_submission.copy()
submission[SUBMISSION_TARGET] = test_predictions

# Save to CSV
submission.to_csv("submission.csv", index=False)
print("Submission file saved!")

submission.head()


## ✅ What’s Next?

This is just a baseline model. You are welcome to improve upon it or design your own pipeline from scratch!

Good luck, and happy modelling! 🚀
